# 📊 Exploratory Data Analysis (EDA) - Student Dropout Early Warning System
**System Focus:** End-of-1st-Semester Early Warning System  
**Dataset:** UCI Student Dropout & Academic Success Dataset (`data/data.csv`)

---

## 🎯 Objectives
1. **Target Analysis:** Inspect target class distribution (`Graduate`, `Dropout`, `Enrolled`).
2. **Demographic & Background Analysis:** Examine student age, admission grades, and prior academic background.
3. **1st Semester Academic Analysis:** Evaluate 1st semester performance metrics (`Curricular units 1st sem approved`, `grade`, etc.).
4. **Bivariate Target Relationships:** Analyze relationships between dropout risk and academic performance, tuition-fee status, scholarship status, and debtor status.
5. **Correlation Heatmap:** Assess correlations among genuinely numerical features.
6. **Data Leakage Safeguard:** Strictly exclude all 2nd semester features (`Curricular units 2nd sem...`).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

# Configure aesthetics
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv("../data/data.csv", sep=";", encoding="utf-8-sig")
df.columns = [c.strip().replace('"', '') for c in df.columns]

print(f"Dataset Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Missing Values: {df.isnull().sum().sum()}")
print(f"Duplicate Rows: {df.duplicated().sum()}")


---
## 1. Feature Categorization & Grouping
- **Target Variable:** `Target` (`Graduate`, `Dropout`, `Enrolled`)
- **Excluded Features (Data Leakage Guard):** All 6 `Curricular units 2nd sem...` features.
- **Categorical Features:** Multi-category integer codes (`Marital status`, `Course`, `Application mode`, etc.)
- **Numerical Features:** Continuous variables, count features, and binary indicators.


In [ ]:
target_col = 'Target'

excluded_2nd_sem = [c for c in df.columns if '2nd sem' in c.lower()]

categorical_cols = [
    'Marital status', 'Application mode', 'Application order', 'Course',
    'Previous qualification', 'Nacionality', "Mother's qualification",
    "Father's qualification", "Mother's occupation", "Father's occupation"
]

binary_cols = [
    'Daytime/evening attendance', 'Displaced', 'Educational special needs',
    'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International'
]

numerical_cols = [
    'Previous qualification (grade)', 'Admission grade', 'Age at enrollment',
    'Unemployment rate', 'Inflation rate', 'GDP',
    'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)',
    'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)'
]

print(f"Target Column: {target_col}")
print(f"Excluded 2nd-Sem Features ({len(excluded_2nd_sem)}): {excluded_2nd_sem}")
print(f"Categorical Features: {len(categorical_cols)}")
print(f"Numerical & Binary Features: {len(numerical_cols) + len(binary_cols)}")


---
### Chart 1: Target Class Distribution


In [ ]:
plt.figure(figsize=(8, 5))
ax = sns.countplot(data=df, x='Target', order=['Graduate', 'Dropout', 'Enrolled'], palette=['#2ecc71', '#e74c3c', '#3498db'])
plt.title('Target Class Distribution', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Target Status', fontsize=12)
plt.ylabel('Count', fontsize=12)

total = len(df)
for p in ax.patches:
    pct = 100 * p.get_height() / total
    x = p.get_x() + p.get_width() / 2
    y = p.get_height() + 30
    ax.annotate(f"{int(p.get_height())} ({pct:.1f}%)", (x, y), ha='center', fontsize=11, fontweight='bold')

plt.ylim(0, max(df['Target'].value_counts()) * 1.15)
plt.tight_layout()
plt.show()


---
### Chart 2: Age at Enrollment Distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='Age at enrollment', kde=True, bins=30, color='#34495e', edgecolor='black')
plt.title('Distribution of Age at Enrollment', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Age at Enrollment', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

mean_age = df['Age at enrollment'].mean()
median_age = df['Age at enrollment'].median()
plt.axvline(mean_age, color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean Age: {mean_age:.1f}')
plt.axvline(median_age, color='#2ecc71', linestyle='-', linewidth=2, label=f'Median Age: {median_age:.0f}')

plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


---
### Chart 3: Admission Grade Distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='Admission grade', kde=True, bins=30, color='#2980b9', edgecolor='black')
plt.title('Distribution of Admission Grades (Range: 0-200 scale)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Admission Grade', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

mean_adm = df['Admission grade'].mean()
plt.axvline(mean_adm, color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean Admission Grade: {mean_adm:.1f}')

plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


---
### Chart 4: Previous Qualification Grade Distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='Previous qualification (grade)', kde=True, bins=30, color='#8e44ad', edgecolor='black')
plt.title('Distribution of Previous Qualification Grades', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Previous Qualification Grade', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

mean_pq = df['Previous qualification (grade)'].mean()
plt.axvline(mean_pq, color='#e74c3c', linestyle='--', linewidth=2, label=f'Mean Grade: {mean_pq:.1f}')

plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


---
### Chart 5: First-Semester Grade Distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(data=df, x='Curricular units 1st sem (grade)', kde=True, bins=30, color='#16a085', edgecolor='black')
plt.title('Distribution of 1st Semester Grades (Range: 0-20 scale)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('1st Semester Grade', fontsize=12)
plt.ylabel('Frequency', fontsize=12)

zero_grades = (df['Curricular units 1st sem (grade)'] == 0).sum()
pct_zero = (zero_grades / len(df)) * 100
plt.annotate(f"Zero grades: {zero_grades} ({pct_zero:.1f}%)", 
             xy=(0.5, 300), xytext=(3, 400),
             arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
             fontsize=11, fontweight='bold', bbox=dict(boxstyle="round,pad=0.3", fc="yellow", ec="b", lw=1))

plt.tight_layout()
plt.show()


---
### Chart 6: First-Semester Approved Units Distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='Curricular units 1st sem (approved)', palette='Blues_r')
plt.title('Distribution of Approved Curricular Units in 1st Semester', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Approved Units Count', fontsize=12)
plt.ylabel('Student Count', fontsize=12)
plt.tight_layout()
plt.show()


---
### Chart 7: Target vs First-Semester Grade


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Target', y='Curricular units 1st sem (grade)', 
            order=['Graduate', 'Dropout', 'Enrolled'], palette=['#2ecc71', '#e74c3c', '#3498db'])
plt.title('1st Semester Grade Distribution across Target Classes', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Target Class', fontsize=12)
plt.ylabel('1st Semester Grade', fontsize=12)

means = df.groupby('Target')['Curricular units 1st sem (grade)'].mean()
print("Mean 1st Sem Grade by Target:")
for t, m in means.items():
    print(f"  {t}: {m:.2f}")

plt.tight_layout()
plt.show()


---
### Chart 8: Target vs Approved Units


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Target', y='Curricular units 1st sem (approved)', 
            order=['Graduate', 'Dropout', 'Enrolled'], palette=['#2ecc71', '#e74c3c', '#3498db'])
plt.title('Approved Units in 1st Semester across Target Classes', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Target Class', fontsize=12)
plt.ylabel('Approved Units (1st Sem)', fontsize=12)

approved_means = df.groupby('Target')['Curricular units 1st sem (approved)'].mean()
print("Mean 1st Sem Approved Units by Target:")
for t, m in approved_means.items():
    print(f"  {t}: {m:.2f}")

plt.tight_layout()
plt.show()


---
### Chart 9: Target vs Tuition-Fee Status


In [ ]:
plt.figure(figsize=(8, 5))
tuition_ct = pd.crosstab(df['Tuition fees up to date'], df['Target'], normalize='index') * 100
tuition_ct = tuition_ct[['Graduate', 'Enrolled', 'Dropout']]

ax = tuition_ct.plot(kind='bar', stacked=True, color=['#2ecc71', '#3498db', '#e74c3c'], figsize=(9, 5))
plt.title('Target Breakdown by Tuition Fees Up-to-Date Status', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Tuition Fees Up to Date (0 = No, 1 = Yes)', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.xticks(ticks=[0, 1], labels=['Not Up-to-Date (0)', 'Up-to-Date (1)'], rotation=0)
plt.legend(title='Target', bbox_to_anchor=(1.02, 1), loc='upper left')

for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if height > 5:
        x, y = p.get_xy() 
        ax.text(x + width/2, y + height/2, f"{height:.1f}%", ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()


---
### Chart 10: Target vs Scholarship Status


In [ ]:
plt.figure(figsize=(8, 5))
schol_ct = pd.crosstab(df['Scholarship holder'], df['Target'], normalize='index') * 100
schol_ct = schol_ct[['Graduate', 'Enrolled', 'Dropout']]

ax = schol_ct.plot(kind='bar', stacked=True, color=['#2ecc71', '#3498db', '#e74c3c'], figsize=(9, 5))
plt.title('Target Breakdown by Scholarship Status', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Scholarship Holder (0 = No, 1 = Yes)', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.xticks(ticks=[0, 1], labels=['Non-Scholarship (0)', 'Scholarship Holder (1)'], rotation=0)
plt.legend(title='Target', bbox_to_anchor=(1.02, 1), loc='upper left')

for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if height > 5:
        x, y = p.get_xy() 
        ax.text(x + width/2, y + height/2, f"{height:.1f}%", ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()


---
### Chart 11: Target vs Debtor Status


In [ ]:
plt.figure(figsize=(8, 5))
debt_ct = pd.crosstab(df['Debtor'], df['Target'], normalize='index') * 100
debt_ct = debt_ct[['Graduate', 'Enrolled', 'Dropout']]

ax = debt_ct.plot(kind='bar', stacked=True, color=['#2ecc71', '#3498db', '#e74c3c'], figsize=(9, 5))
plt.title('Target Breakdown by Debtor Status', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Debtor (0 = No, 1 = Yes)', fontsize=12)
plt.ylabel('Percentage (%)', fontsize=12)
plt.xticks(ticks=[0, 1], labels=['Non-Debtor (0)', 'Debtor (1)'], rotation=0)
plt.legend(title='Target', bbox_to_anchor=(1.02, 1), loc='upper left')

for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if height > 5:
        x, y = p.get_xy() 
        ax.text(x + width/2, y + height/2, f"{height:.1f}%", ha='center', va='center', color='white', fontweight='bold')

plt.tight_layout()
plt.show()


---
### Chart 12: Correlation Heatmap for Genuinely Numerical Features
Note: Excludes nominal categorical codes (`Course`, `Mother's occupation`, etc.) and 2nd semester features.


In [ ]:
plt.figure(figsize=(12, 10))
corr_cols = numerical_cols + binary_cols
corr_matrix = df[corr_cols].corr()

sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Correlation Heatmap (Genuinely Numerical & Binary Features)', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()
